1. Installez les bibliothèques requises

In [17]:
import torch
import spacy
import sklearn

# Check if gensim is installed, if not, install it
try:
    import gensim
except ImportError:
    print("Gensim non trouvé. Installation de gensim...")
    !pip install gensim
    import gensim # Réessayer d'importer après l'installation

print(f"PyTorch : {torch.__version__}")
print(f"spaCy   : {spacy.__version__}")
print(f"scikit-learn : {sklearn.__version__}")
print(f"Gensim  : {gensim.__version__}")
print(f"CUDA disponible : {torch.cuda.is_available()}") # Pour vérifier le support GPU

PyTorch : 2.11.0+cpu
spaCy   : 3.8.14
scikit-learn : 1.6.1
Gensim  : 4.4.0
CUDA disponible : False


In [14]:
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_layer_size, output_size, dropout_rate=0.2):
        super().__init__()
        self.hidden_layer_size = hidden_layer_size

        self.lstm = nn.LSTM(
            input_size,
            hidden_layer_size,
            num_layers=1,
            batch_first=True # Input and output tensors are provided as (batch, seq, feature)
        )

        self.dropout = nn.Dropout(dropout_rate)

        self.linear = nn.Linear(hidden_layer_size, output_size)

    def forward(self, input_seq):
        # Initialize hidden state and cell state for the LSTM within forward pass
        # This handles dynamic batching correctly as it's called for each batch
        # input_seq is (batch_size, input_size). We need to reshape it to (batch_size, seq_len=1, input_size) for LSTM.
        # The LSTM output will be (batch_size, seq_len, hidden_size). We take the last (and only) sequence step.
        h_0 = torch.zeros(1, input_seq.size(0), self.hidden_layer_size).to(input_seq.device)
        c_0 = torch.zeros(1, input_seq.size(0), self.hidden_layer_size).to(input_seq.device)

        lstm_out, _ = self.lstm(input_seq.unsqueeze(1), (h_0, c_0))

        # Apply dropout and pass through linear layer
        # lstm_out[:, -1, :] takes the last output of the sequence for each item in the batch
        predictions = self.linear(self.dropout(lstm_out[:, -1, :]))
        return predictions

In [21]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# 2. CHARGER ET PRÉTRAITER L'ENSEMBLE DE DONNÉES


# Simulation du chargement d'un dataset boursier (Remplacez par votre fichier)
# Le fichier doit contenir au moins une colonne 'Close'
np_data = np.sin(np.linspace(0, 100, 1000)) * 50 + 100 + np.random.normal(0, 2, 1000)
df = pd.DataFrame({'Close': np_data, 'Volume': np.random.rand(1000) * 100000})

# Création de la colonne cible : prix de clôture du lendemain
df['Target'] = df['Close'].shift(-1)

# Suppression de la dernière ligne car sa cible sera NaN
df = df.dropna().reset_index(drop=True)

# Suppression des colonnes inutiles (Exemple: on ne garde que Close pour prédire Target)
features = df[['Close']].values
target = df[['Target']].values

# Normalisation avec MinMaxScaler
scaler_features = MinMaxScaler(feature_range=(0, 1))
scaler_target = MinMaxScaler(feature_range=(0, 1))

features_scaled = scaler_features.fit_transform(features)
target_scaled = scaler_target.fit_transform(target)

In [24]:
# 3. PRÉPARER L'ENSEMBLE DE DONNÉES POUR L'ENTRAÎNEMENT

import torch
from torch.utils.data import Dataset, DataLoader

# Définition de la fenêtre temporelle (Lookback)
def create_sequences(features, target, seq_length=30):
    X, y = [], []
    for i in range(len(features) - seq_length):
        X.append(features[i:i+seq_length])
        y.append(target[i+seq_length])
    return torch.tensor(np.array(X), dtype=torch.float32), torch.tensor(np.array(y), dtype=torch.float32)

LOOKBACK = 30
X, y = create_sequences(features_scaled, target_scaled, seq_length=LOOKBACK)

# Division : 70% Entraînement, 15% Validation, 15% Test
train_size = int(len(X) * 0.70)
val_size = int(len(X) * 0.15)

X_train, y_train = X[:train_size], y[:train_size]
X_val, y_val = X[train_size:train_size+val_size], y[train_size:train_size+val_size]
X_test, y_test = X[train_size+val_size:], y[train_size+val_size:]

# Classe personnalisée PyTorch Dataset
class StockDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Création des DataLoaders
BATCH_SIZE = 32
train_loader = DataLoader(StockDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(StockDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(StockDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

In [26]:
# 4. DÉFINIR LE MODÈLE LSTM
import torch.nn as nn

class StockLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, dropout_prob=0.2):
        super(StockLSTM, self).__init__()
        # Remarque : Votre énoncé mentionne d'inclure des couches GRU dans l'architecture LSTM,
        # nous utilisons donc nn.GRU ici comme demandé implicitement pour capturer les séquences.
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout_prob)
        self.dropout = nn.Dropout(dropout_prob)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x shape: (batch_size, seq_length, input_size)
        out, _ = self.gru(x)
        # Prendre la dernière sortie de la séquence temporelle
        out = out[:, -1, :]
        out = self.dropout(out)
        out = self.fc(out)
        return out

# Initialisation du modèle
model = StockLSTM(input_size=1, hidden_size=64, num_layers=2, dropout_prob=0.2)

In [29]:
# 5. ENTRAÎNER LE MODÈLE

import torch.optim as optim
import torch.nn as nn

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 20

for epoch in range(EPOCHS):
    # Mode Entraînement
    model.train()
    train_loss = 0.0
    for inputs, targets in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * inputs.size(0)

    # Mode Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, targets in val_loader:
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            val_loss += loss.item() * inputs.size(0)

    train_loss /= len(train_loader.dataset)
    val_loss /= len(val_loader.dataset)

    print(f"Époque {epoch+1:02d}/{EPOCHS} | Perte Entraînement : {train_loss:.5f} | Perte Validation : {val_loss:.5f}")

Époque 01/20 | Perte Entraînement : 0.00402 | Perte Validation : 0.00057
Époque 02/20 | Perte Entraînement : 0.00262 | Perte Validation : 0.00050
Époque 03/20 | Perte Entraînement : 0.00278 | Perte Validation : 0.00117
Époque 04/20 | Perte Entraînement : 0.00286 | Perte Validation : 0.00128
Époque 05/20 | Perte Entraînement : 0.00320 | Perte Validation : 0.00081
Époque 06/20 | Perte Entraînement : 0.00243 | Perte Validation : 0.00062
Époque 07/20 | Perte Entraînement : 0.00262 | Perte Validation : 0.00047
Époque 08/20 | Perte Entraînement : 0.00272 | Perte Validation : 0.00048
Époque 09/20 | Perte Entraînement : 0.00220 | Perte Validation : 0.00050
Époque 10/20 | Perte Entraînement : 0.00232 | Perte Validation : 0.00067
Époque 11/20 | Perte Entraînement : 0.00301 | Perte Validation : 0.00149
Époque 12/20 | Perte Entraînement : 0.00265 | Perte Validation : 0.00058
Époque 13/20 | Perte Entraînement : 0.00243 | Perte Validation : 0.00061
Époque 14/20 | Perte Entraînement : 0.00243 | Perte

In [31]:
# 6. ÉVALUER LE MODÈLE ET SAUVEGARDER LE SCALER

from sklearn.metrics import r2_score
import joblib

model.eval()
all_predictions = []
all_targets = []

with torch.no_grad():
    for inputs, targets in test_loader:
        outputs = model(inputs)
        all_predictions.append(outputs.numpy())
        all_targets.append(targets.numpy())

# Regroupement des résultats de test
all_predictions = np.vstack(all_predictions)
all_targets = np.vstack(all_targets)

# Inverser la normalisation pour obtenir les vrais prix en dollars / euros
true_predictions = scaler_target.inverse_transform(all_predictions)
true_targets = scaler_target.inverse_transform(all_targets)

# Calcul du score R²
r2 = r2_score(true_targets, true_predictions)
print(f"\nScore R² final sur l'ensemble de test : {r2:.4f}")

# Sauvegarde des fichiers requis pour les prédictions futures
torch.save(model.state_dict(), 'stock_lstm_model.pth')
joblib.dump(scaler_target, 'target_scaler.pkl')
joblib.dump(scaler_features, 'features_scaler.pkl')
print("Modèle et objets scalers sauvegardés avec succès !")


Score R² final sur l'ensemble de test : 0.9919
Modèle et objets scalers sauvegardés avec succès !
